# InterviewIQ TF-IDF reference baseline (separate from the Mohler model)

This notebook fits a TF-IDF vectorizer to the project's reference answers and combines text overlap with key-concept coverage. It is an exploratory baseline, not a supervised grader trained on Mohler score labels, and it is not used by the deployed Mohler API in `backend/app.py`.

In [ ]:
!pip -q install scikit-learn joblib
import json, re, joblib
from pathlib import Path
from google.colab import files
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


## Load the project question bank

In [ ]:
uploaded = files.upload()  # Select questions.json from InterviewIQ_Complete
bank_path = next((name for name in uploaded if name.endswith('.json')), None)
if bank_path is None:
    raise ValueError('Upload questions.json to continue')
bank = json.loads(Path(bank_path).read_text(encoding='utf-8'))
rows = [row for row in bank if row.get('reference_answer')]
references = [row['reference_answer'] for row in rows]
print(f'Loaded {len(rows)} reference answers')


## Fit and export the scoring model

The vectorizer learns vocabulary and phrase weights from the project's reference answers. Scoring combines semantic similarity with reference keyword and supplied key-concept coverage.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
vectorizer.fit(references)
artifact = {'vectorizer': vectorizer, 'version': 1, 'training_rows': len(references)}
joblib.dump(artifact, 'ml_artifacts.joblib')
print('Saved ml_artifacts.joblib')


In [ ]:
def score_answer(answer, reference_answer, key_concepts=()):
    answer = answer.strip()
    if not answer or not reference_answer.strip():
        return {'score': 0, 'semantic_similarity': 0, 'concept_coverage': 0, 'keyword_coverage': 0}
    vec = vectorizer.transform([answer, reference_answer])
    semantic = float((vec[0] @ vec[1].T).toarray()[0, 0])
    concepts = [c for c in key_concepts if c.strip()]
    concept = sum(c.lower() in answer.lower() for c in concepts) / len(concepts) if concepts else 0.0
    tokens = lambda text: set(re.findall(r'[a-z0-9+#.]+', text.lower()))
    ref_words, answer_words = tokens(reference_answer), tokens(answer)
    keyword = len(ref_words & answer_words) / len(ref_words) if ref_words else 0.0
    score = round(100 * (0.55 * semantic + 0.25 * concept + 0.20 * keyword))
    return {'score': max(0, min(100, score)), 'semantic_similarity': round(semantic, 3), 'concept_coverage': round(concept, 3), 'keyword_coverage': round(keyword, 3)}

example = rows[0]
score_answer(example['reference_answer'], example['reference_answer'], example.get('key_concepts', []))


## Adaptive next-question difficulty

The service smooths the current score with up to two previous scores. Strong recent performance moves up one available level; a low average moves down one; otherwise it stays at the current level. The browser selects an unused question at that level and falls back to the closest available level for the topic.

In [ ]:
LEVELS = ['Easy', 'Medium', 'Hard']
def recommend_difficulty(score, current='Medium', recent_scores=()):
    history = [max(0, min(100, float(x))) for x in list(recent_scores)[-2:]] + [score]
    average = sum(history) / len(history)
    index = LEVELS.index(current) if current in LEVELS else 1
    if average >= 78: index = min(2, index + 1)
    elif average < 42: index = max(0, index - 1)
    return LEVELS[index]
print('Example next level:', recommend_difficulty(86, 'Medium', [80, 75]))


## Download the TF-IDF baseline artifact

This artifact belongs only to this exploratory baseline. It is not compatible with `prepai_mohler_model.joblib` or the Mohler API adapter.

In [ ]:
files.download('ml_artifacts.joblib')
